# ORION: Train the `Hey Orion` Wake Word

This Colab notebook follows the official `openWakeWord` v0.6.0 automatic trainer. It never edits the Raspberry Pi or production Orion configuration. It does not download replacement feature models, and it will not report a trained model unless the trainer completes successfully, the expected ONNX file exists, and post-training validation passes.

**Inputs to upload from this repository:** `artifacts/wakeword-staging/melspectrogram.onnx`, `artifacts/wakeword-staging/embedding_model.onnx`, `src/orion/audio/wakeword_assets.py`, and `src/orion/config.py`.

**Data rights gate:** the official openWakeWord feature dataset declares CC BY-NC-SA 4.0. The Piper sample-generator checkpoint is based on LibriTTS-R under CC BY 4.0. Review and explicitly acknowledge those terms in the configuration cell before downloading or training. Unknown-license room impulse responses are not used.

## 1. Pin Runtime, Tag, and Provenance

Use a fresh Linux GPU Colab runtime. The source and dataset revisions below are immutable. This notebook records the actual environment and package freeze. Training is not bit-for-bit deterministic across GPU/driver versions; the sources, settings, random seeds, outputs, and hashes are recorded to make the run auditable and repeatable.

In [ ]:
from __future__ import annotations

import ast
import csv
import hashlib
import json
import os
import platform
import random
import shutil
import subprocess
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np

WORKSPACE = Path("/workspace")
PROVENANCE_DIR = WORKSPACE / "provenance"
PROVENANCE_DIR.mkdir(parents=True, exist_ok=True)
RUN_LOG = PROVENANCE_DIR / "run_log.txt"
PROVENANCE_PATH = PROVENANCE_DIR / "provenance.json"
OPENWAKEWORD_REPO = "https://github.com/dscripka/openWakeWord.git"
OPENWAKEWORD_COMMIT = "c8ef6912c5feccf1037b852d9bc6c7ed644135ba"
PIPER_REPO = "https://github.com/rhasspy/piper-sample-generator.git"
PIPER_COMMIT = "195e3bd967d54589c2137c9de2b22ad526ba6b6f"
FEATURE_DATASET = "davidscripka/openwakeword_features"
FEATURE_DATASET_REVISION = "985bf1b47e7f19c07741af82bfe32d5a9dc56096"
FEATURE_SHA256 = {
    "melspectrogram.onnx": "ba2b0e0f8b7b875369a2c89cb13360ff53bac436f2895cced9f479fa65eb176f",
    "embedding_model.onnx": "70d164290c1d095d1d4ee149bc5e00543250a7316b59f31d056cff7bd3075c1f",
}
TARGET_PHRASE = "Hey Orion"
TARGET_COUNT = 20_000
VALIDATION_POSITIVE_COUNT = 4_000
RANDOM_SEED = 20261006


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def write_json(path: Path, value: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, indent=2, sort_keys=True) + "\n", encoding="utf-8")


def git_commit(directory: Path) -> str:
    return subprocess.check_output(["git", "-C", str(directory), "rev-parse", "HEAD"], text=True).strip()

provenance = {
    "record_type": "orion-hey-orion-training-run",
    "started_utc": datetime.now(timezone.utc).isoformat(),
    "python": sys.version,
    "platform": platform.platform(),
    "runtime": "Google Colab Linux; record GPU/runtime details in subsequent cells",
    "target_phrase": TARGET_PHRASE,
    "target_count": TARGET_COUNT,
    "validation_positive_count": VALIDATION_POSITIVE_COUNT,
    "random_seed": RANDOM_SEED,
    "repositories": {},
    "datasets": [],
    "assets": [],
    "license_acknowledgements": {},
}
write_json(PROVENANCE_PATH, provenance)
RUN_LOG.write_text("ORION Hey Orion wake-word training run\n", encoding="utf-8")
print(f"UTC start: {provenance['started_utc']}")
print(f"Python: {sys.version.split()[0]}; platform: {platform.platform()}")

## 2. Fetch the Official v0.6.0 Workflow and the Staged Feature Models

The official training notebook and trainer are cloned at the tag's commit. This cell also saves the upstream notebook for reference and clones ORION read-only. The official feature graphs are **not** downloaded here: upload the verified local files in the next cell.

In [ ]:
OPENWAKEWORD_DIR = WORKSPACE / "openWakeWord"
PIPER_DIR = WORKSPACE / "piper-sample-generator"
ORION_REPO_URL = "https://github.com/HarishPotnuri/ORION.git"
ORION_SOURCE_DIR = WORKSPACE / "orion-source-read-only"

if not OPENWAKEWORD_DIR.exists():
    subprocess.run(["git", "clone", OPENWAKEWORD_REPO, str(OPENWAKEWORD_DIR)], check=True)
subprocess.run(["git", "-C", str(OPENWAKEWORD_DIR), "checkout", "--detach", OPENWAKEWORD_COMMIT], check=True)
assert git_commit(OPENWAKEWORD_DIR) == OPENWAKEWORD_COMMIT

if not PIPER_DIR.exists():
    subprocess.run(["git", "clone", PIPER_REPO, str(PIPER_DIR)], check=True)
subprocess.run(["git", "-C", str(PIPER_DIR), "checkout", "--detach", PIPER_COMMIT], check=True)
assert git_commit(PIPER_DIR) == PIPER_COMMIT

if not ORION_SOURCE_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", ORION_REPO_URL, str(ORION_SOURCE_DIR)], check=True)

upstream_notebook_url = (
    "https://raw.githubusercontent.com/dscripka/openWakeWord/"
    "v0.6.0/notebooks/automatic_model_training.ipynb"
)
notebook_bytes = subprocess.check_output([sys.executable, "-c", "import urllib.request,sys; sys.stdout.buffer.write(urllib.request.urlopen(sys.argv[1]).read())", upstream_notebook_url])
(PROVENANCE_DIR / "official_automatic_model_training_v0.6.0.ipynb").write_bytes(notebook_bytes)
provenance["repositories"].update({
    "openwakeword": {"url": OPENWAKEWORD_REPO, "tag": "v0.6.0", "commit": git_commit(OPENWAKEWORD_DIR)},
    "piper_sample_generator": {"url": PIPER_REPO, "tag": "v2.0.0", "commit": git_commit(PIPER_DIR)},
    "orion_reference_clone": {"url": ORION_REPO_URL, "commit": git_commit(ORION_SOURCE_DIR), "mode": "read-only reference"},
})
write_json(PROVENANCE_PATH, provenance)
print("openWakeWord:", git_commit(OPENWAKEWORD_DIR))
print("Piper sample generator:", git_commit(PIPER_DIR))
print("ORION reference clone is read-only; no config files are modified.")

In [ ]:
# Install graph inspection/runtime packages before the feature-upload validation cell.
subprocess.check_call([sys.executable, "-m", "pip", "install", "onnx>=1.14,<2", "onnxruntime>=1.10,<2"])
import onnx
import onnxruntime as ort
print("ONNX inspection:", onnx.__version__, "ORT:", ort.__version__)

In [ ]:
from google.colab import files
import onnx
import onnxruntime as ort

print("Upload these exact files from artifacts/wakeword-staging:", ", ".join(FEATURE_SHA256))
feature_uploads = files.upload()
FEATURE_DIR = OPENWAKEWORD_DIR / "openwakeword" / "resources" / "models"
FEATURE_DIR.mkdir(parents=True, exist_ok=True)
for filename, expected_sha in FEATURE_SHA256.items():
    if filename not in feature_uploads:
        raise RuntimeError(f"Required staged local feature model not uploaded: {filename}")
    local_path = WORKSPACE / filename
    local_path.write_bytes(feature_uploads[filename])
    actual_sha = sha256_file(local_path)
    if actual_sha != expected_sha:
        raise RuntimeError(f"SHA-256 mismatch for {filename}: {actual_sha}")
    model = onnx.load(str(local_path))
    onnx.checker.check_model(model)
    shutil.copy2(local_path, FEATURE_DIR / filename)
    print(f"{filename}: ONNX checker PASS; sha256={actual_sha}")

mel_session = ort.InferenceSession(str(FEATURE_DIR / "melspectrogram.onnx"), providers=["CPUExecutionProvider"])
embedding_session = ort.InferenceSession(str(FEATURE_DIR / "embedding_model.onnx"), providers=["CPUExecutionProvider"])
assert mel_session.get_inputs()[0].name == "input"
assert len(mel_session.get_inputs()[0].shape) == 2
assert len(mel_session.get_outputs()[0].shape) == 4 and mel_session.get_outputs()[0].shape[-1] == 32
assert embedding_session.get_inputs()[0].name == "input_1"
assert embedding_session.get_inputs()[0].shape[1:] == [76, 32, 1]
assert len(embedding_session.get_outputs()[0].shape) == 4 and embedding_session.get_outputs()[0].shape[-1] == 96
provenance["assets"].extend([
    {"file": name, "sha256": FEATURE_SHA256[name], "source": "official openWakeWord v0.5.1 release", "license": "CC-BY-NC-SA-4.0 conservative upstream model-artifact interpretation"}
    for name in FEATURE_SHA256
])
write_json(PROVENANCE_PATH, provenance)
print("Feature graph signatures verified on ONNX Runtime CPU.")

## 3. Inspect Orion's Validator Contract (Read-Only)

Upload the current local `src/orion/audio/wakeword_assets.py` and `src/orion/config.py` from this checkout. The remote clone is retained only as a read-only reference; parsing the uploaded files ensures this report reflects local changes. The parser records actual tensor requirements, manifest keys, configured names/paths, and the CLI entry point. No cell in this notebook opens or writes `/etc/orion/orion.env` or any Pi path.

In [ ]:
local_orion_uploads = files.upload()
required_local = {"wakeword_assets.py", "config.py"}
if not required_local.issubset(local_orion_uploads):
    raise RuntimeError(f"Upload current local Orion files: {sorted(required_local)}")
ORION_SNAPSHOT = PROVENANCE_DIR / "local_orion_snapshot"
ORION_SNAPSHOT.mkdir(parents=True, exist_ok=True)
for filename in required_local:
    (ORION_SNAPSHOT / filename).write_bytes(local_orion_uploads[filename])

validator_path = ORION_SNAPSHOT / "wakeword_assets.py"
config_path = ORION_SNAPSHOT / "config.py"
validator_text = validator_path.read_text(encoding="utf-8")
config_text = config_path.read_text(encoding="utf-8")
validator_tree = ast.parse(validator_text)
config_tree = ast.parse(config_text)

def module_assignment(tree, variable):
    for node in tree.body:
        if isinstance(node, (ast.Assign, ast.AnnAssign)):
            targets = node.targets if isinstance(node, ast.Assign) else [node.target]
            if any(isinstance(target, ast.Name) and target.id == variable for target in targets):
                return ast.literal_eval(node.value)
    return None

expected_runtime = module_assignment(validator_tree, "_EXPECTED_RUNTIME_VERSION")
expected_compatibility = module_assignment(validator_tree, "_EXPECTED_COMPATIBILITY")
expected_assets = module_assignment(validator_tree, "_ASSETS")
tensor_contracts = []
for node in ast.walk(validator_tree):
    if isinstance(node, ast.Call) and isinstance(node.func, ast.Name) and node.func.id == "_check_tensor":
        kwargs = {item.arg: item.value for item in node.keywords}
        tensor_contracts.append({
            "label": ast.literal_eval(kwargs["label"]),
            "name": ast.literal_eval(kwargs["name"]),
            "rank": ast.literal_eval(kwargs["rank"]),
            "shape": ast.literal_eval(kwargs["shape"]),
        })
manifest_fields = sorted(set(re.findall(r'manifest\.get\(["\']([^"\']+)', validator_text)))
threshold_match = re.search(r'wake_word_threshold\s*=\s*_env_float\([^,]+,\s*([0-9.]+)\)', config_text)
ORION_WAKEWORD_MODEL_DIR = ORION_SOURCE_DIR / "artifacts" / "wakeword-staging" / "hey_orion_v0_6_0"
ORION_DEFAULT_PI_MODEL_DIR = "/opt/orion/models/wakeword"

contract_report = {
    "runtime_version": expected_runtime,
    "expected_asset_keys": list(expected_assets or []),
    "manifest_top_level_keys": manifest_fields,
    "compatibility_constants": expected_compatibility,
    "tensor_contracts": tensor_contracts,
    "wake_threshold_setting": float(threshold_match.group(1)) if threshold_match else None,
    "manifest_has_threshold_field": "threshold" in manifest_fields,
    "output_node_name_required": False,
    "output_dtype": "tensor(float)",
    "classifier_input": {"rank": 3, "shape": ["batch", "positive_fixed_feature_frames", 96]},
    "classifier_output": {"rank": 3, "shape": ["batch", 1, 1], "confidence_range": [0, 1]},
    "model_filenames": ["hey_orion.onnx", "melspectrogram.onnx", "embedding_model.onnx", "phase13-artifacts.json"],
    "default_pi_model_dir": ORION_DEFAULT_PI_MODEL_DIR,
    "orion_repository_staging_dir": str(ORION_WAKEWORD_MODEL_DIR),
    "validator_cli": "python -m orion --validate-wakeword-assets (paths supplied via temporary ORION_* environment variables)",
    "local_validator_sha256": sha256_file(validator_path),
    "local_config_sha256": sha256_file(config_path),
}
assert expected_runtime == "0.6.0"
assert set(expected_assets) == {"wakeword", "melspectrogram", "embedding"}
assert not any(str(path).lower().startswith(("/etc/orion", "/opt/orion")) for path in [WORKSPACE / "exports"])
write_json(PROVENANCE_DIR / "contract_report.json", contract_report)
provenance["orion_contract_report_sha256"] = sha256_file(PROVENANCE_DIR / "contract_report.json")
write_json(PROVENANCE_PATH, provenance)
print(json.dumps(contract_report, indent=2))

## 4. Create the Reproducible Colab Workspace

All notebook outputs go under `/workspace`. The deterministic tree keeps generated positives, negatives, approved noise/RIR (if any), validation, manifests, licenses, training models, evaluation, provenance, and downloadable exports separate. No destination is under `/etc/orion` or `/opt/orion`.

In [ ]:
WORKSPACE_DIRS = [
    WORKSPACE / "config",
    WORKSPACE / "data" / "positive",
    WORKSPACE / "data" / "negative",
    WORKSPACE / "data" / "noise",
    WORKSPACE / "data" / "rir",
    WORKSPACE / "data" / "validation",
    WORKSPACE / "manifests",
    WORKSPACE / "models",
    WORKSPACE / "eval",
    WORKSPACE / "licenses",
    WORKSPACE / "provenance",
    WORKSPACE / "exports",
]
for directory in WORKSPACE_DIRS:
    directory.mkdir(parents=True, exist_ok=True)
assert all(directory.is_dir() for directory in WORKSPACE_DIRS)
assert WORKSPACE.resolve() in (WORKSPACE / "exports").resolve().parents
assert "/etc/orion" not in str((WORKSPACE / "exports").resolve()).replace("\\", "/")
assert "/opt/orion" not in str((WORKSPACE / "exports").resolve()).replace("\\", "/")

def print_tree(root: Path, max_depth: int = 3) -> None:
    for path in sorted(root.rglob("*")):
        relative = path.relative_to(root)
        if len(relative.parts) <= max_depth:
            print("  " * (len(relative.parts) - 1) + (relative.name + ("/" if path.is_dir() else "")))

print_tree(WORKSPACE)
print("Required directories verified:", len(WORKSPACE_DIRS))

## 5. Define the Exact Target Phrase

The target is one phrase only: `Hey Orion` (case-preserved, English US). No alternate wake phrases or aliases are added. openWakeWordâ€™s text-to-speech and pronunciation tools may internally normalize text for synthesis, but the training configuration and phrase metadata retain this exact target string.

In [ ]:
phrase_path = WORKSPACE / "config" / "phrase.txt"
phrase_path.write_text(TARGET_PHRASE, encoding="utf-8")
phrase_metadata = {
    "training_target_id": "orion_hey_orion_openwakeword_v0_6_0",
    "phrase": TARGET_PHRASE,
    "language": "English",
    "locale": "en-US",
    "phrase_casing": "Hey Orion",
    "aliases": [],
    "target_phrase_config": [TARGET_PHRASE],
}
write_json(WORKSPACE / "config" / "metadata.json", phrase_metadata)
assert phrase_path.read_text(encoding="utf-8") == "Hey Orion"
assert phrase_metadata["target_phrase_config"] == ["Hey Orion"]
assert phrase_metadata["aliases"] == []
print("Exact phrase target verified:", repr(phrase_path.read_text(encoding="utf-8")))

## 6. Positive Synthetic Samples (20,000 Minimum)

The official Piper sample generator produces synthetic, multi-speaker positive clips from the exact target phrase. `n_samples` is fixed at 20,000 and `n_samples_val` is 4,000, held out by the official pipeline. The checkpoint source is the Piper v2.0.0 `en_US-libritts_r-medium.pt` release; its LibriTTS-R model card states CC BY 4.0. The MIT-licensed generator code and CC BY 4.0 checkpoint require attribution. A later cell requires an explicit acknowledgement before any checkpoint download.

In [ ]:
assert TARGET_PHRASE == "Hey Orion"
assert TARGET_COUNT >= 20_000
assert VALIDATION_POSITIVE_COUNT > 0
TRAINING_SEED = RANDOM_SEED
POSITIVE_MANIFEST = WORKSPACE / "manifests" / "positive_manifest.csv"
with POSITIVE_MANIFEST.open("w", newline="", encoding="utf-8") as stream:
    writer = csv.DictWriter(stream, fieldnames=["split", "file", "sha256", "phrase", "sample_rate_hz", "channels", "source", "revision", "license", "speaker_variation", "seed"])
    writer.writeheader()

print({
    "target_phrase": TARGET_PHRASE,
    "training_positive_target": TARGET_COUNT,
    "heldout_positive_target": VALIDATION_POSITIVE_COUNT,
    "seed": TRAINING_SEED,
    "positive_tts": "official Piper sample generator v2.0.0 / LibriTTS-R medium; CC BY 4.0",
})

## 7. Negative and Adversarial Data (License-Gated)

The official trainer creates phonetic adversarial negatives from `Hey Orion` with Piper; these are not alternate wake targets. Generic negative features and the approximately 11.3-hour false-positive validation features come from `davidscripka/openwakeword_features`, pinned to revision `985bf1b47e7f19c07741af82bfe32d5a9dc56096`. Its dataset card declares CC BY-NC-SA 4.0 and identifies ACAV100M plus DiPCo, Santa Barbara spoken-corpus, and MUSDB-derived validation features. The material is noncommercial/share-alike only. Do not run the acknowledgement gate for commercial use.

In [ ]:
# Change these only after reviewing the linked dataset cards and confirming the terms fit your use.
ACK_CC_BY_NC_SA_NONCOMMERCIAL = False
ACK_CC_BY_4_ATTRIBUTION = False
if not ACK_CC_BY_NC_SA_NONCOMMERCIAL:
    raise RuntimeError("Stop: review CC BY-NC-SA 4.0 terms for staged feature models and openWakeWord negative/validation features, then explicitly acknowledge noncommercial/share-alike use.")
if not ACK_CC_BY_4_ATTRIBUTION:
    raise RuntimeError("Stop: review CC BY 4.0 attribution obligations for the Piper LibriTTS-R checkpoint, then explicitly acknowledge.")

provenance["license_acknowledgements"] = {
    "CC-BY-NC-SA-4.0": {"accepted": True, "scope": "noncommercial, attribution, share-alike", "dataset_revision": FEATURE_DATASET_REVISION},
    "CC-BY-4.0": {"accepted": True, "scope": "attribution for Piper LibriTTS-R checkpoint"},
}
write_json(WORKSPACE / "licenses" / "license_summary.json", provenance["license_acknowledgements"])
license_summary = (
    "Training uses openWakeWord feature-model release artifacts and the official generic-negative/false-positive feature dataset under a conservative CC BY-NC-SA 4.0 interpretation. "
    "Piper sample-generator source is MIT; the LibriTTS-R medium checkpoint identifies its source data as CC BY 4.0. "
    "No commercial use is authorized by CC BY-NC-SA. No third-party raw noise, AudioSet/FMA audio, or unknown-license RIR files are downloaded."
)
(WORKSPACE / "licenses" / "license_summary.md").write_text(license_summary + "\n", encoding="utf-8")
print("License acknowledgements recorded. Continue only for noncommercial use under these terms.")

## 8. Background and Noise Data

No raw background-audio corpus is downloaded. The official augmentation adds generated colored noise and uses the licensed general-negative embedding features above. The `background_paths` configuration is empty, so no YouTube clips, AudioSet, FMA, or other third-party recordings enter the run. This is recorded as a limitation, not hidden.

In [ ]:
noise_manifest_path = WORKSPACE / "manifests" / "noise_manifest.csv"
with noise_manifest_path.open("w", newline="", encoding="utf-8") as stream:
    writer = csv.DictWriter(stream, fieldnames=["asset", "role", "source", "revision", "license", "sha256", "usage"])
    writer.writeheader()
    writer.writerow({
        "asset": "openWakeWord generated colored noise augmentation",
        "role": "synthetic noise mixed during feature augmentation",
        "source": f"openWakeWord {OPENWAKEWORD_COMMIT}",
        "revision": OPENWAKEWORD_COMMIT,
        "license": "Apache-2.0 code; no external noise file",
        "sha256": "not-applicable-generated-during-run",
        "usage": "augmentation only; no raw external background path configured",
    })
background_paths = []
assert background_paths == []
print("No external raw background audio will be downloaded or used.")

## 9. Room Impulse Responses (RIR)

The upstream notebook points to `davidscripka/MIT_environmental_impulse_responses`, but its Hugging Face dataset card declares `license: unknown`. It is therefore excluded. `rir_paths` stays empty. The model can still be trained with the official augmentations and noise features, but this run does not establish reverberation robustness; use a separately reviewed/licensed RIR source before making far-field claims.

In [ ]:
rir_paths = []
rir_manifest_path = WORKSPACE / "manifests" / "rir_manifest.csv"
with rir_manifest_path.open("w", newline="", encoding="utf-8") as stream:
    writer = csv.DictWriter(stream, fieldnames=["asset", "source", "revision", "license", "sha256", "decision"])
    writer.writeheader()
    writer.writerow({
        "asset": "MIT_environmental_impulse_responses",
        "source": "https://huggingface.co/datasets/davidscripka/MIT_environmental_impulse_responses",
        "revision": "b824a1ef2821f112fda0b9cb26e4278c62b425bb",
        "license": "unknown (dataset card)",
        "sha256": "",
        "decision": "excluded; not downloaded or used",
    })
assert rir_paths == []
print("RIR inputs disabled because the cited upstream dataset license is unknown.")

## 10. Validation Data and Split Manifests

The official feature dataset revision supplies two separate files: ACAV100M-derived generic negative training features and the curated false-positive validation features. The openWakeWord trainer separately generates held-out positive/adversarial clips (`n_samples_val=4,000`) for balanced false-negative/false-positive evaluation. A feature-source revision and SHA-256 are recorded for each split. Generated positive train/test audio is hash-deduplicated after generation. The feature dataset declares its splits by source corpus; it contains no raw audio to independently re-deduplicate.

In [ ]:
subprocess.check_call([sys.executable, "-m", "pip", "install", "huggingface_hub"])
from huggingface_hub import HfApi, hf_hub_download

required_free_bytes = 30 * 1024**3
free_bytes = shutil.disk_usage(WORKSPACE).free
if free_bytes < required_free_bytes:
    raise RuntimeError(f"Need at least 30 GiB free for the pinned 17.5 GB feature repository and training outputs; found {free_bytes / 1024**3:.1f} GiB.")
api_revision = HfApi().dataset_info(FEATURE_DATASET, revision=FEATURE_DATASET_REVISION).sha
assert api_revision == FEATURE_DATASET_REVISION

feature_data_dir = WORKSPACE / "data" / "negative"
train_feature_path = Path(hf_hub_download(
    repo_id=FEATURE_DATASET,
    filename="openwakeword_features_ACAV100M_2000_hrs_16bit.npy",
    repo_type="dataset",
    revision=FEATURE_DATASET_REVISION,
    local_dir=str(feature_data_dir),
))
validation_feature_path = Path(hf_hub_download(
    repo_id=FEATURE_DATASET,
    filename="validation_set_features.npy",
    repo_type="dataset",
    revision=FEATURE_DATASET_REVISION,
    local_dir=str(WORKSPACE / "data" / "validation"),
))
train_sha = sha256_file(train_feature_path)
validation_sha = sha256_file(validation_feature_path)
train_features = np.load(train_feature_path, mmap_mode="r")
validation_features = np.load(validation_feature_path, mmap_mode="r")
assert train_features.ndim == 3 and train_features.shape[1:] == (16, 96)
assert validation_features.ndim == 2 and validation_features.shape[1] == 96
assert train_features.dtype == np.float32 and validation_features.dtype == np.float32

negative_manifest_path = WORKSPACE / "manifests" / "negative_manifest.csv"
with negative_manifest_path.open("w", newline="", encoding="utf-8") as stream:
    writer = csv.DictWriter(stream, fieldnames=["asset", "role", "dataset", "revision", "license", "sha256", "usage_scope"])
    writer.writeheader()
    writer.writerow({"asset": train_feature_path.name, "role": "generic negative training features", "dataset": FEATURE_DATASET, "revision": FEATURE_DATASET_REVISION, "license": "CC-BY-NC-SA-4.0", "sha256": train_sha, "usage_scope": "noncommercial model training and redistribution under share-alike"})

validation_manifest_path = WORKSPACE / "manifests" / "validation_manifest.csv"
with validation_manifest_path.open("w", newline="", encoding="utf-8") as stream:
    writer = csv.DictWriter(stream, fieldnames=["asset", "split", "dataset", "revision", "license", "sha256", "hours_approx", "usage_scope"])
    writer.writeheader()
    writer.writerow({"asset": validation_feature_path.name, "split": "false-positive-validation", "dataset": FEATURE_DATASET, "revision": FEATURE_DATASET_REVISION, "license": "CC-BY-NC-SA-4.0", "sha256": validation_sha, "hours_approx": 11.3, "usage_scope": "noncommercial validation only"})

feature_records = [
    {"dataset": FEATURE_DATASET, "revision": FEATURE_DATASET_REVISION, "file": train_feature_path.name, "sha256": train_sha, "license": "CC-BY-NC-SA-4.0", "role": "generic negative training"},
    {"dataset": FEATURE_DATASET, "revision": FEATURE_DATASET_REVISION, "file": validation_feature_path.name, "sha256": validation_sha, "license": "CC-BY-NC-SA-4.0", "role": "false-positive validation"},
]
provenance["datasets"].extend(feature_records)
write_json(PROVENANCE_PATH, provenance)
print("Generic negative features:", train_features.shape, train_sha)
print("False-positive validation features:", validation_features.shape, validation_sha)
print("Pinned HF revision:", api_revision)

## 11. Install and Freeze the Training/Validation Packages

The upstream notebook's required training stack is installed at its documented versions. Core openWakeWord runtime dependencies and ONNX/ORT CPU validation dependencies are installed separately. The official trainer's final TFLite conversion is omitted by a narrowly audited AST transformation: Orion consumes ONNX only. The exact resolved versions are frozen to `provenance/requirements_colab.lock` and copied into exports.

In [ ]:
core_requirements = [
    "onnxruntime>=1.10,<2",
    "onnx>=1.14,<2",
    "scipy>=1.3,<2",
    "scikit-learn>=1,<2",
    "requests>=2,<3",
    "tqdm>=4,<5",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", *core_requirements])
subprocess.check_call([sys.executable, "-m", "pip", "install", "--no-deps", "-e", str(OPENWAKEWORD_DIR)])

upstream_training_requirements = [
    "mutagen==1.47.0",
    "torchinfo==1.8.0",
    "torchmetrics==1.2.0",
    "speechbrain==0.5.14",
    "audiomentations==0.33.0",
    "torch-audiomentations==0.11.0",
    "acoustics==0.2.6",
    "PyYAML>=6.0,<7",
    "pronouncing==0.2.0",
    "deep-phonemizer==0.0.19",
    "datasets==2.14.6",
    "piper-phonemize",
    "webrtcvad==2.0.10",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", *upstream_training_requirements])

import onnx
import onnxruntime
import pronouncing
import torch

missing_pronunciations = [word for word in TARGET_PHRASE.lower().split() if not pronouncing.phones_for_word(word)]
if missing_pronunciations:
    raise RuntimeError(f"Stop: upstream adversarial generator would download an unpinned phonemizer for {missing_pronunciations}; review and pin that model/license before continuing.")

freeze_text = subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True)
lock_path = PROVENANCE_DIR / "requirements_colab.lock"
lock_path.write_text(freeze_text, encoding="utf-8")
provenance["python_packages_lock_sha256"] = sha256_file(lock_path)
provenance["runtime_versions"] = {
    "python": sys.version.split()[0],
    "torch": torch.__version__,
    "onnx": onnx.__version__,
    "onnxruntime": onnxruntime.__version__,
}
write_json(PROVENANCE_PATH, provenance)
print(json.dumps(provenance["runtime_versions"], indent=2))
print("Frozen environment:", lock_path)

In [ ]:
gpu_result = subprocess.run(["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"], capture_output=True, text=True)
if not torch.cuda.is_available() or gpu_result.returncode != 0:
    raise RuntimeError("Select a Linux GPU runtime in Colab before training; this notebook does not silently fall back to long CPU training.")
provenance["runtime_versions"]["cuda_available"] = True
provenance["runtime_versions"]["gpu"] = gpu_result.stdout.strip()
write_json(PROVENANCE_PATH, provenance)
print("Recorded Colab GPU:", gpu_result.stdout.strip())

## 12. Configure and Run the Official v0.6.0 Training Workflow

The upstream YAML is the starting point. Only the exact target/counts, paths, negative-feature filename, and explicit empty raw-background/RIR lists are configured here. Positive synthesis and adversarial negatives are generated by the pinned Piper/openWakeWord code. Training runs as three checked stages; ONNX is the required output. A patched copy of the pinned CLI removes only its optional TFLite conversion call, because Orion's contract is ONNX-only.

In [ ]:
import yaml

PIPER_CHECKPOINT_URL = "https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt"
PIPER_CHECKPOINT_PATH = PIPER_DIR / "models" / "en_US-libritts_r-medium.pt"
PIPER_CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)
if not PIPER_CHECKPOINT_PATH.is_file():
    import urllib.request
    urllib.request.urlretrieve(PIPER_CHECKPOINT_URL, PIPER_CHECKPOINT_PATH)
piper_checkpoint_sha = sha256_file(PIPER_CHECKPOINT_PATH)

base_config_path = OPENWAKEWORD_DIR / "examples" / "custom_model.yml"
with base_config_path.open(encoding="utf-8") as stream:
    training_config = yaml.safe_load(stream)
training_config.update({
    "model_name": "hey_orion",
    "target_phrase": ["Hey Orion"],
    "custom_negative_phrases": [],
    "n_samples": 20_000,
    "n_samples_val": 4_000,
    "tts_batch_size": 50,
    "augmentation_batch_size": 16,
    "piper_sample_generator_path": str(PIPER_DIR),
    "output_dir": str(WORKSPACE / "data"),
    "rir_paths": [],
    "background_paths": [],
    "background_paths_duplication_rate": [],
    "false_positive_validation_data_path": str(validation_feature_path),
    "feature_data_files": {"ACAV100M_sample": str(train_feature_path)},
    "batch_n_per_class": {"ACAV100M_sample": 1024, "adversarial_negative": 50, "positive": 50},
})
assert training_config["target_phrase"] == ["Hey Orion"]
assert training_config["n_samples"] >= 20_000
assert training_config["piper_sample_generator_path"] == str(PIPER_DIR)
assert training_config["feature_data_files"]["ACAV100M_sample"] == str(train_feature_path)

TRAINING_CONFIG_PATH = WORKSPACE / "config" / "hey_orion_training.yaml"
TRAINING_CONFIG_PATH.write_text(yaml.safe_dump(training_config, sort_keys=False), encoding="utf-8")
provenance["assets"].append({
    "file": PIPER_CHECKPOINT_PATH.name,
    "source": PIPER_CHECKPOINT_URL,
    "revision": PIPER_COMMIT,
    "license": "CC-BY-4.0 (LibriTTS-R model card); Piper generator code MIT",
    "sha256": piper_checkpoint_sha,
})
provenance["training_config_sha256"] = sha256_file(TRAINING_CONFIG_PATH)
write_json(PROVENANCE_PATH, provenance)
print(TRAINING_CONFIG_PATH.read_text(encoding="utf-8"))
print("Piper checkpoint SHA-256:", piper_checkpoint_sha)

In [ ]:
trainer_source_path = OPENWAKEWORD_DIR / "openwakeword" / "train.py"
trainer_source = trainer_source_path.read_text(encoding="utf-8")
trainer_tree = ast.parse(trainer_source)

class RemoveOptionalTFLiteExport(ast.NodeTransformer):
    def __init__(self):
        self.removed = 0
    def visit_Expr(self, node):
        value = node.value
        if isinstance(value, ast.Call) and isinstance(value.func, ast.Name) and value.func.id == "convert_onnx_to_tflite":
            self.removed += 1
            return None
        return self.generic_visit(node)

converter_patch = RemoveOptionalTFLiteExport()
trainer_tree = converter_patch.visit(trainer_tree)
ast.fix_missing_locations(trainer_tree)
if converter_patch.removed != 1:
    raise RuntimeError(f"Expected exactly one optional TFLite export call in pinned trainer; found {converter_patch.removed}. Stop and inspect upstream source.")
TRAINER_ONNX_ONLY_PATH = OPENWAKEWORD_DIR / "openwakeword" / "train_onnx_only.py"
TRAINER_ONNX_ONLY_PATH.write_text(ast.unparse(trainer_tree) + "\n", encoding="utf-8")
provenance["trainer_source_sha256"] = hashlib.sha256(trainer_source.encode("utf-8")).hexdigest()
provenance["trainer_patch"] = "AST-only deletion of the single optional convert_onnx_to_tflite call; training/generation/export_model implementation unchanged"
provenance["trainer_onnx_only_sha256"] = sha256_file(TRAINER_ONNX_ONLY_PATH)
write_json(PROVENANCE_PATH, provenance)

class Tee:
    def __init__(self, *streams):
        self.streams = streams
    def write(self, value):
        for stream in self.streams:
            stream.write(value)
            stream.flush()
    def flush(self):
        for stream in self.streams:
            stream.flush()

def run_official_stage(flag: str) -> None:
    if flag not in {"--generate_clips", "--augment_clips", "--train_model"}:
        raise ValueError(f"Unexpected trainer stage: {flag}")
    random.seed(TRAINING_SEED)
    np.random.seed(TRAINING_SEED)
    torch.manual_seed(TRAINING_SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(TRAINING_SEED)
    old_argv = sys.argv
    sys.argv = [str(TRAINER_ONNX_ONLY_PATH), "--training_config", str(TRAINING_CONFIG_PATH), flag]
    try:
        with RUN_LOG.open("a", encoding="utf-8") as log_stream:
            tee = Tee(sys.stdout, log_stream)
            from contextlib import redirect_stdout, redirect_stderr
            with redirect_stdout(tee), redirect_stderr(tee):
                import runpy
                runpy.run_path(str(TRAINER_ONNX_ONLY_PATH), run_name="__main__")
    finally:
        sys.argv = old_argv

print("Patched only optional TFLite export; pinned ONNX trainer remains intact.")

### Execute the Three Official Stages

The training claim is intentionally gated. The completion marker is written only if generation and augmentation return, the trainer completes, and the ONNX artifact exists. Failure at any stage stops the notebook.

In [ ]:
run_official_stage("--generate_clips")
model_data_dir = WORKSPACE / "data" / "hey_orion"
positive_train_dir = model_data_dir / "positive_train"
positive_test_dir = model_data_dir / "positive_test"
negative_train_dir = model_data_dir / "negative_train"
negative_test_dir = model_data_dir / "negative_test"
positive_train_files = sorted(positive_train_dir.glob("*.wav"))
positive_test_files = sorted(positive_test_dir.glob("*.wav"))
if len(positive_train_files) < TARGET_COUNT:
    raise RuntimeError(f"Positive train target not met: {len(positive_train_files)} < {TARGET_COUNT}. Do not proceed.")
if len(positive_test_files) < VALIDATION_POSITIVE_COUNT:
    raise RuntimeError(f"Held-out positive target not met: {len(positive_test_files)} < {VALIDATION_POSITIVE_COUNT}. Do not proceed.")

positive_rows = []
train_hashes = set()
for split, paths in (("positive-train", positive_train_files), ("positive-validation", positive_test_files)):
    current_hashes = set()
    for path in paths:
        digest = sha256_file(path)
        current_hashes.add(digest)
        positive_rows.append({
            "split": split,
            "file": str(path.relative_to(WORKSPACE)),
            "sha256": digest,
            "phrase": TARGET_PHRASE,
            "sample_rate_hz": 16000,
            "channels": 1,
            "source": PIPER_CHECKPOINT_URL,
            "revision": PIPER_COMMIT,
            "license": "CC-BY-4.0 (LibriTTS-R checkpoint source)",
            "speaker_variation": "official Piper multi-speaker synthesis and configured length scales",
            "seed": TRAINING_SEED,
        })
    if split == "positive-train":
        train_hashes = current_hashes
    else:
        overlap = train_hashes & current_hashes
        if overlap:
            raise RuntimeError(f"Positive train/validation audio hash leakage: {len(overlap)} duplicate files")
with POSITIVE_MANIFEST.open("a", newline="", encoding="utf-8") as stream:
    writer = csv.DictWriter(stream, fieldnames=positive_rows[0].keys())
    writer.writerows(positive_rows)

run_official_stage("--augment_clips")
negative_rows = []
for split, directory in (("adversarial-train", negative_train_dir), ("adversarial-validation", negative_test_dir)):
    for path in sorted(directory.glob("*.wav")):
        negative_rows.append({
            "asset": str(path.relative_to(WORKSPACE)),
            "role": split,
            "dataset": "openWakeWord generate_adversarial_texts + Piper sample generator",
            "revision": OPENWAKEWORD_COMMIT,
            "license": "openWakeWord code Apache-2.0; Piper synthetic source CC-BY-4.0",
            "sha256": sha256_file(path),
            "usage_scope": "adversarial negative only; not an alternate target phrase",
        })
with negative_manifest_path.open("a", newline="", encoding="utf-8") as stream:
    writer = csv.DictWriter(stream, fieldnames=negative_rows[0].keys())
    writer.writerows(negative_rows)

run_official_stage("--train_model")
TRAINED_MODEL_PATH = WORKSPACE / "data" / "hey_orion.onnx"
if not TRAINED_MODEL_PATH.is_file() or TRAINED_MODEL_PATH.stat().st_size == 0:
    raise RuntimeError("Trainer returned without a non-empty ONNX classifier; model is not trained")
completion = {
    "training_completed": True,
    "phrase": TARGET_PHRASE,
    "positive_train_count": len(positive_train_files),
    "positive_validation_count": len(positive_test_files),
    "adversarial_train_count": len(list(negative_train_dir.glob("*.wav"))),
    "adversarial_validation_count": len(list(negative_test_dir.glob("*.wav"))),
    "classifier_path": str(TRAINED_MODEL_PATH),
    "classifier_bytes": TRAINED_MODEL_PATH.stat().st_size,
    "classifier_sha256": sha256_file(TRAINED_MODEL_PATH),
    "completed_utc": datetime.now(timezone.utc).isoformat(),
}
write_json(WORKSPACE / "provenance" / "training_completion.json", completion)
provenance["training_completion"] = completion
write_json(PROVENANCE_PATH, provenance)
print("TRAINING COMPLETION MARKER WRITTEN:", json.dumps(completion, indent=2))

In [ ]:
import wave

wav_inventory = {}
for split_name, paths in (
    ("positive-train", positive_train_files),
    ("positive-validation", positive_test_files),
    ("adversarial-train", sorted(negative_train_dir.glob("*.wav"))),
    ("adversarial-validation", sorted(negative_test_dir.glob("*.wav"))),
):
    for path in paths:
        with wave.open(str(path), "rb") as wav_file:
            header = (wav_file.getnchannels(), wav_file.getsampwidth(), wav_file.getframerate(), wav_file.getnframes())
        if header[0:3] != (1, 2, 16000) or header[3] <= 0:
            raise RuntimeError(f"Expected non-empty 16kHz mono PCM16 WAV for {split_name}: {path} header={header}")
    wav_inventory[split_name] = len(paths)
provenance["generated_wav_inventory"] = wav_inventory
write_json(PROVENANCE_PATH, provenance)
print("Verified PCM WAV sample rate/channels/bit depth:", wav_inventory)

In [ ]:
raw_classifier_sha = sha256_file(TRAINED_MODEL_PATH)
classifier_graph = onnx.load(str(TRAINED_MODEL_PATH))
if len(classifier_graph.graph.output) != 1:
    raise RuntimeError(f"Expected one classifier output; found {len(classifier_graph.graph.output)}")
raw_output = classifier_graph.graph.output[0]
raw_shape = [dim.dim_value if dim.HasField("dim_value") else (dim.dim_param or None) for dim in raw_output.type.tensor_type.shape.dim]
output_adapter = "none; trainer output already matched Orion rank-3 contract"

if len(raw_shape) == 2 and raw_shape[1] == 1:
    original_output_name = raw_output.name
    flat_name = original_output_name + "__flat"
    reshaped_name = original_output_name + "__orion_rank3"
    shape_initializer_name = "orion_classifier_output_shape"
    for node in classifier_graph.graph.node:
        for index, name in enumerate(node.output):
            if name == original_output_name:
                node.output[index] = flat_name
    for value_info in classifier_graph.graph.value_info:
        if value_info.name == original_output_name:
            value_info.name = flat_name
    classifier_graph.graph.initializer.append(
        onnx.numpy_helper.from_array(np.asarray([-1, 1, 1], dtype=np.int64), name=shape_initializer_name)
    )
    classifier_graph.graph.node.append(
        onnx.helper.make_node("Reshape", [flat_name, shape_initializer_name], [reshaped_name], name="OrionOutputRankAdapter")
    )
    raw_output.name = reshaped_name
    output_shape = raw_output.type.tensor_type.shape
    output_shape.ClearField("dim")
    output_shape.dim.add().dim_param = "batch"
    output_shape.dim.add().dim_value = 1
    output_shape.dim.add().dim_value = 1
    classifier_graph = onnx.shape_inference.infer_shapes(classifier_graph)
    for output in classifier_graph.graph.output:
        if output.name == reshaped_name:
            output.type.tensor_type.elem_type = onnx.TensorProto.FLOAT
            shape = output.type.tensor_type.shape
            shape.ClearField("dim")
            shape.dim.add().dim_param = "batch"
            shape.dim.add().dim_value = 1
            shape.dim.add().dim_value = 1
    output_adapter = "ONNX Reshape from [batch,1] to [batch,1,1], preserving values and dynamic batch"
elif len(raw_shape) == 3 and raw_shape[1:] == [1, 1]:
    pass
else:
    raise RuntimeError(f"Trainer output is incompatible and not safely adaptable: {raw_shape}")

adapted_temp_path = TRAINED_MODEL_PATH.with_suffix(".adapted.onnx")
onnx.checker.check_model(classifier_graph)
onnx.save(classifier_graph, str(adapted_temp_path))
check_session = ort.InferenceSession(str(adapted_temp_path), providers=["CPUExecutionProvider"])
check_input = check_session.get_inputs()[0]
check_output = check_session.get_outputs()[0]
check_shape = check_input.shape
if len(check_shape) != 3 or check_shape[-1] != 96 or not isinstance(check_shape[1], int) or check_shape[1] <= 0:
    raise RuntimeError(f"Classifier input does not satisfy Orion feature contract: {check_shape}")
if len(check_output.shape) != 3 or check_output.shape[1:] != [1, 1] or check_output.type != "tensor(float)":
    raise RuntimeError(f"Classifier output does not satisfy Orion contract: {check_output.shape} {check_output.type}")
smoke = check_session.run(None, {check_input.name: np.zeros((1, check_shape[1], 96), dtype=np.float32)})[0]
assert smoke.shape == (1, 1, 1) and np.isfinite(smoke).all() and (smoke >= 0).all() and (smoke <= 1).all()
adapted_temp_path.replace(TRAINED_MODEL_PATH)
completion["classifier_pre_adapter_sha256"] = raw_classifier_sha
completion["output_adapter"] = output_adapter
completion["classifier_sha256"] = sha256_file(TRAINED_MODEL_PATH)
completion["classifier_bytes"] = TRAINED_MODEL_PATH.stat().st_size
write_json(completion_path, completion)
provenance["training_completion"] = completion
provenance["output_adapter"] = output_adapter
write_json(PROVENANCE_PATH, provenance)
print("Observed raw output:", raw_shape)
print("Applied adapter:", output_adapter)
print("Orion-compatible output:", check_output.shape, check_output.type)

In [ ]:
# Rewrite provenance manifests with consistent columns and an explicit license-source URL for every external asset.
OPENWAKEWORD_LICENSE_URL = "https://github.com/dscripka/openWakeWord/blob/v0.6.0/README.md#license"
FEATURE_DATASET_LICENSE_URL = "https://huggingface.co/datasets/davidscripka/openwakeword_features"
PIPER_CHECKPOINT_LICENSE_URL = "https://huggingface.co/datasets/rhasspy/piper-checkpoints/tree/main/en/en_US/libritts_r/medium"
PIPER_CODE_LICENSE_URL = "https://github.com/rhasspy/piper-sample-generator/blob/v2.0.0/LICENSE.md"

for asset in provenance["assets"]:
    if asset.get("file") == "melspectrogram.onnx":
        asset.update({"revision": "1eec2158c5c54150ac5f4c15065adacb1003b1e7", "license_source": OPENWAKEWORD_LICENSE_URL})
    elif asset.get("file") == "embedding_model.onnx":
        asset.update({"revision": "1eec2158c5c54150ac5f4c15065adacb1003b1e7", "license_source": OPENWAKEWORD_LICENSE_URL})
    elif asset.get("file") == PIPER_CHECKPOINT_PATH.name:
        asset["license_source"] = PIPER_CHECKPOINT_LICENSE_URL
        asset["generator_code_license_source"] = PIPER_CODE_LICENSE_URL
for dataset in provenance["datasets"]:
    if dataset.get("dataset") == FEATURE_DATASET:
        dataset["license_source"] = FEATURE_DATASET_LICENSE_URL

positive_fields = ["split", "file", "sha256", "phrase", "sample_rate_hz", "channels", "source", "revision", "license", "license_source", "speaker_variation", "seed"]
with POSITIVE_MANIFEST.open("w", newline="", encoding="utf-8") as stream:
    writer = csv.DictWriter(stream, fieldnames=positive_fields)
    writer.writeheader()
    for row in positive_rows:
        row["license_source"] = PIPER_CHECKPOINT_LICENSE_URL
        writer.writerow(row)

negative_fields = ["asset", "role", "dataset", "revision", "license", "license_source", "sha256", "usage_scope"]
negative_final_rows = [{
    "asset": train_feature_path.name,
    "role": "generic negative training features",
    "dataset": FEATURE_DATASET,
    "revision": FEATURE_DATASET_REVISION,
    "license": "CC-BY-NC-SA-4.0",
    "license_source": FEATURE_DATASET_LICENSE_URL,
    "sha256": train_sha,
    "usage_scope": "noncommercial training under attribution/share-alike terms",
}]
for row in negative_rows:
    negative_final_rows.append({
        "asset": row["asset"],
        "role": row["role"],
        "dataset": row["dataset"],
        "revision": row["revision"],
        "license": row["license"],
        "license_source": PIPER_CHECKPOINT_LICENSE_URL,
        "sha256": row["sha256"],
        "usage_scope": row["usage_scope"],
    })
with negative_manifest_path.open("w", newline="", encoding="utf-8") as stream:
    writer = csv.DictWriter(stream, fieldnames=negative_fields)
    writer.writeheader()
    writer.writerows(negative_final_rows)

validation_fields = ["asset", "split", "dataset", "revision", "license", "license_source", "sha256", "hours_approx", "usage_scope"]
with validation_manifest_path.open("w", newline="", encoding="utf-8") as stream:
    writer = csv.DictWriter(stream, fieldnames=validation_fields)
    writer.writeheader()
    writer.writerow({"asset": validation_feature_path.name, "split": "false-positive-validation", "dataset": FEATURE_DATASET, "revision": FEATURE_DATASET_REVISION, "license": "CC-BY-NC-SA-4.0", "license_source": FEATURE_DATASET_LICENSE_URL, "sha256": validation_sha, "hours_approx": 11.3, "usage_scope": "noncommercial validation only"})
write_json(PROVENANCE_PATH, provenance)
print("Final train/validation manifests include source revisions, checksums and license references.")

### Adapt the Observed Classifier Output to Orion's Runtime Shape

The pinned trainer's binary export may be rank 2 `[batch,1]`; Orion and openWakeWord's score extraction require `[batch,1,1]`. The next cell inspects the actual ONNX graph. It preserves an already-compatible rank-3 output, wraps only the exact rank-2 `[batch,1]` case in an ONNX `Reshape` with inferred batch dimension, and fails on every other shape.

## 13. Export Classifier, Orion Manifest, Checksums, and Provenance

Only continue after `training_completion.json` exists and confirms the 20,000/4,000 sample minimums. Export names and manifest keys below match ORION's current validator. The shared feature graphs are described by their existing local SHA-256 values; they are not regenerated.

In [ ]:
completion_path = WORKSPACE / "provenance" / "training_completion.json"
if not completion_path.is_file():
    raise RuntimeError("Training completion marker is missing; do not create/export a model package")
completion = json.loads(completion_path.read_text(encoding="utf-8"))
if not completion.get("training_completed") or completion.get("positive_train_count", 0) < 20_000 or completion.get("positive_validation_count", 0) < 4_000:
    raise RuntimeError("Training completion marker does not meet requested minimum counts")

MODEL_EXPORT_DIR = WORKSPACE / "models"
MODEL_EXPORT_DIR.mkdir(exist_ok=True)
EXPORT_MODEL_PATH = MODEL_EXPORT_DIR / "hey_orion.onnx"
shutil.copy2(TRAINED_MODEL_PATH, EXPORT_MODEL_PATH)
model_sha = sha256_file(EXPORT_MODEL_PATH)
onnx_model = onnx.load(str(EXPORT_MODEL_PATH))


def value_info_shape(value_info):
    dims = []
    for dim in value_info.type.tensor_type.shape.dim:
        dims.append(dim.dim_value if dim.HasField("dim_value") else (dim.dim_param or None))
    return dims

classifier_input_info = onnx_model.graph.input[0]
classifier_output_info = onnx_model.graph.output[0]
classifier_input_shape = value_info_shape(classifier_input_info)
classifier_output_shape = value_info_shape(classifier_output_info)
assert len(classifier_input_shape) == 3 and classifier_input_shape[-1] == 96
assert isinstance(classifier_input_shape[1], int) and classifier_input_shape[1] > 0
assert len(classifier_output_shape) == 3 and classifier_output_shape[1:] == [1, 1]

metadata = {
    "wake_phrase": TARGET_PHRASE,
    "model_name": "hey_orion",
    "runtime": {"name": "openwakeword", "version": "0.6.0", "framework": "onnx"},
    "sample_rate_hz": 16000,
    "threshold": 0.5,
    "threshold_source": "ORION default ORION_WAKE_WORD_THRESHOLD; tune only after evaluation",
    "classifier_input": {"name": classifier_input_info.name, "type": "tensor(float)", "shape": classifier_input_shape},
    "classifier_output": {"name": classifier_output_info.name, "type": "tensor(float)", "shape": classifier_output_shape},
    "sha256": model_sha,
    "training_completion": completion,
}
write_json(MODEL_EXPORT_DIR / "hey_orion_metadata.json", metadata)

manifest = {
    "schema_version": 1,
    "wake_phrase": "Hey Orion",
    "sample_rate_hz": 16000,
    "runtime": {"name": "openwakeword", "version": "0.6.0", "framework": "onnx"},
    "compatibility": {
        "sample_rate_hz": 16000,
        "melspectrogram_bins": 32,
        "embedding_window_frames": 76,
        "embedding_dimension": 96,
        "wake_output_classes": 1,
        "feature_model_sha256": {
            "melspectrogram": FEATURE_SHA256["melspectrogram.onnx"],
            "embedding": FEATURE_SHA256["embedding_model.onnx"],
        },
    },
    "assets": {
        "wakeword": {
            "file": "hey_orion.onnx",
            "sha256": model_sha,
            "source": "trained in this Colab run; see provenance.json and training_config.yaml",
            "revision": OPENWAKEWORD_COMMIT,
            "license": "CC-BY-NC-SA-4.0 conservative training-data terms; review before redistribution",
            "training_recipe": f"Official openWakeWord v0.6.0 automatic trainer at {OPENWAKEWORD_COMMIT}; target phrase exact; {TARGET_COUNT} positive synthetic train clips; seed {TRAINING_SEED}; ONNX-only export",
            "training_data": f"Piper LibriTTS-R medium checkpoint CC-BY-4.0 positives; {FEATURE_DATASET}@{FEATURE_DATASET_REVISION} CC-BY-NC-SA-4.0 generic negatives and false-positive validation; no raw background or RIR datasets",
        },
        "melspectrogram": {
            "file": "melspectrogram.onnx",
            "sha256": FEATURE_SHA256["melspectrogram.onnx"],
            "source": "https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.onnx",
            "revision": "1eec2158c5c54150ac5f4c15065adacb1003b1e7",
            "license": "CC-BY-NC-SA-4.0 conservative upstream model-artifact interpretation",
        },
        "embedding": {
            "file": "embedding_model.onnx",
            "sha256": FEATURE_SHA256["embedding_model.onnx"],
            "source": "https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.onnx",
            "revision": "1eec2158c5c54150ac5f4c15065adacb1003b1e7",
            "license": "CC-BY-NC-SA-4.0 conservative upstream model-artifact interpretation",
        },
    },
}
write_json(MODEL_EXPORT_DIR / "phase13-artifacts.json", manifest)
shutil.copy2(TRAINING_CONFIG_PATH, MODEL_EXPORT_DIR / "training_config.yaml")
shutil.copy2(PROVENANCE_PATH, MODEL_EXPORT_DIR / "provenance.json")
print("Exported candidate classifier; not yet validated:", EXPORT_MODEL_PATH, model_sha)
print("Input contract:", classifier_input_shape, "output:", classifier_output_shape)

## 14. Post-Training Validation: ONNX, ORT CPU, Contract, Features, FP/FN

This gate must pass before packaging. It checks the classifier and both staged feature graphs, invokes the mel-to-embedding-to-classifier chain on CPU, loads the classifier through openWakeWord itself, computes false-negative rates on held-out synthetic positives/adversarial negatives, and estimates false-positive events/hour on the pinned ~11.3-hour feature validation set. These are dataset metrics only; they do not certify the Pi microphone or room.

In [ ]:
def ort_shape(value_info):
    return list(value_info.shape)

feature_sessions = {
    "mel": ort.InferenceSession(str(FEATURE_DIR / "melspectrogram.onnx"), providers=["CPUExecutionProvider"]),
    "embedding": ort.InferenceSession(str(FEATURE_DIR / "embedding_model.onnx"), providers=["CPUExecutionProvider"]),
}
classifier_session = ort.InferenceSession(str(EXPORT_MODEL_PATH), providers=["CPUExecutionProvider"])
for path in (FEATURE_DIR / "melspectrogram.onnx", FEATURE_DIR / "embedding_model.onnx", EXPORT_MODEL_PATH):
    onnx.checker.check_model(onnx.load(str(path)))

mel_input = feature_sessions["mel"].get_inputs()[0]
mel_output = feature_sessions["mel"].get_outputs()[0]
embedding_input = feature_sessions["embedding"].get_inputs()[0]
embedding_output = feature_sessions["embedding"].get_outputs()[0]
classifier_input = classifier_session.get_inputs()[0]
classifier_output = classifier_session.get_outputs()[0]
assert mel_input.name == "input" and mel_input.type == "tensor(float)" and len(ort_shape(mel_input)) == 2
assert mel_output.type == "tensor(float)" and len(ort_shape(mel_output)) == 4 and ort_shape(mel_output)[-1] == 32
assert embedding_input.name == "input_1" and ort_shape(embedding_input)[1:] == [76, 32, 1]
assert len(ort_shape(embedding_output)) == 4 and ort_shape(embedding_output)[-1] == 96
assert classifier_input.type == "tensor(float)" and len(ort_shape(classifier_input)) == 3 and ort_shape(classifier_input)[-1] == 96
assert isinstance(ort_shape(classifier_input)[1], int) and ort_shape(classifier_input)[1] > 0
assert classifier_output.type == "tensor(float)" and len(ort_shape(classifier_output)) == 3
assert ort_shape(classifier_output)[1:] == [1, 1]

mel_raw = feature_sessions["mel"].run(None, {mel_input.name: np.zeros((1, 64_000), dtype=np.float32)})[0]
assert mel_raw.shape[0] == 1 and mel_raw.shape[1] == 1 and mel_raw.shape[2] >= 76 and mel_raw.shape[3] == 32
mel_features = np.squeeze(mel_raw[0]) / 10 + 2
embedding_windows = np.stack([mel_features[i:i+76] for i in range(0, len(mel_features), 8) if len(mel_features[i:i+76]) == 76])
embedding_raw = feature_sessions["embedding"].run(None, {embedding_input.name: embedding_windows[..., None].astype(np.float32)})[0]
assert embedding_raw.shape[1:] == (1, 1, 96)
embedding_features = np.squeeze(embedding_raw, axis=(1, 2))
classifier_frames = ort_shape(classifier_input)[1]
assert len(embedding_features) >= classifier_frames
chain_score = classifier_session.run(None, {classifier_input.name: embedding_features[-classifier_frames:][None].astype(np.float32)})[0]
assert chain_score.shape == (1, 1, 1)
assert np.isfinite(chain_score).all() and ((chain_score >= 0).all() and (chain_score <= 1).all())

from openwakeword.model import Model as OpenWakeWordModel
runtime_model = OpenWakeWordModel(
    wakeword_models=[str(EXPORT_MODEL_PATH)],
    inference_framework="onnx",
    melspec_model_path=str(FEATURE_DIR / "melspectrogram.onnx"),
    embedding_model_path=str(FEATURE_DIR / "embedding_model.onnx"),
    ncpu=1,
)
runtime_scores = [runtime_model.predict(np.zeros(1280, dtype=np.int16))["hey_orion"] for _ in range(8)]
assert all(np.isfinite(runtime_scores)) and all(0 <= score <= 1 for score in runtime_scores)

THRESHOLD = 0.5

def sequence_max_score(sequence):
    if sequence.ndim != 2 or sequence.shape[1] != 96 or sequence.shape[0] < classifier_frames:
        raise ValueError(f"Expected feature sequence (frames, 96) with >= {classifier_frames} frames; got {sequence.shape}")
    maxima = []
    for offset in range(0, sequence.shape[0] - classifier_frames + 1, 2048):
        ends = min(sequence.shape[0] - classifier_frames + 1, offset + 2048)
        batch = np.stack([sequence[i:i+classifier_frames] for i in range(offset, ends)]).astype(np.float32)
        scores = classifier_session.run(None, {classifier_input.name: batch})[0].reshape(-1)
        maxima.append(scores)
    return float(np.max(np.concatenate(maxima)))

positive_test_features = np.load(model_data_dir / "positive_features_test.npy", mmap_mode="r")
negative_test_features = np.load(model_data_dir / "negative_features_test.npy", mmap_mode="r")
positive_clip_scores = np.array([sequence_max_score(positive_test_features[i]) for i in range(len(positive_test_features))])
negative_clip_scores = np.array([sequence_max_score(negative_test_features[i]) for i in range(len(negative_test_features))])
fn_count = int((positive_clip_scores < THRESHOLD).sum())
negative_accept_count = int((negative_clip_scores >= THRESHOLD).sum())

# Scan consecutive feature windows from the official ~11.3-hour false-positive validation set.
validation_hits = []
for offset in range(0, len(validation_features) - classifier_frames + 1, 2048):
    stop = min(len(validation_features) - classifier_frames + 1, offset + 2048)
    batch = np.stack([validation_features[i:i+classifier_frames] for i in range(offset, stop)]).astype(np.float32)
    scores = classifier_session.run(None, {classifier_input.name: batch})[0].reshape(-1)
    validation_hits.extend((scores >= THRESHOLD).tolist())
hits = np.asarray(validation_hits, dtype=bool)
hit_indexes = np.flatnonzero(hits)
min_gap_frames = int(0.5 / 0.08)
events = 0
previous = -min_gap_frames - 1
for index in hit_indexes:
    if index - previous > min_gap_frames:
        events += 1
    previous = index
validation_hours = len(validation_features) * 0.08 / 3600
metrics = {
    "threshold": THRESHOLD,
    "synthetic_positive_validation_clips": int(len(positive_clip_scores)),
    "false_negative_clips": fn_count,
    "false_negative_rate": float(fn_count / max(1, len(positive_clip_scores))),
    "adversarial_negative_validation_clips": int(len(negative_clip_scores)),
    "adversarial_false_accept_clips": negative_accept_count,
    "adversarial_false_accept_rate": float(negative_accept_count / max(1, len(negative_clip_scores))),
    "general_negative_validation_feature_frames": int(len(hits)),
    "general_negative_threshold_hit_windows": int(hits.sum()),
    "general_negative_threshold_hit_window_rate": float(hits.mean()) if len(hits) else 0.0,
    "general_negative_validation_hours_derived": validation_hours,
    "clustered_false_accept_events": int(events),
    "clustered_false_accept_events_per_hour": float(events / max(validation_hours, 1e-9)),
    "pi_room_acoustic_evaluation": "not performed; synthetic/data-feature metrics only",
}
write_json(WORKSPACE / "eval" / "evaluation.json", metrics)
validation_report = {
    "onnx_checker": "PASS",
    "onnxruntime_cpu": "PASS",
    "classifier_input": {"name": classifier_input.name, "type": classifier_input.type, "shape": ort_shape(classifier_input)},
    "classifier_output": {"name": classifier_output.name, "type": classifier_output.type, "shape": ort_shape(classifier_output)},
    "mel_raw_output_shape": list(mel_raw.shape),
    "embedding_raw_output_shape": list(embedding_raw.shape),
    "feature_chain_smoke_score": float(chain_score.reshape(-1)[0]),
    "openwakeword_runtime_smoke": "PASS",
    "feature_model_sha256": FEATURE_SHA256,
    "classifier_sha256": model_sha,
    "evaluation": metrics,
}
write_json(WORKSPACE / "eval" / "validation_report.json", validation_report)
print(json.dumps(validation_report, indent=2))

In [ ]:
import importlib.util
validator_spec = importlib.util.spec_from_file_location("orion_wakeword_assets", ORION_SNAPSHOT / "wakeword_assets.py")
if validator_spec is None or validator_spec.loader is None:
    raise RuntimeError("Could not load uploaded Orion validator source")
orion_validator = importlib.util.module_from_spec(validator_spec)
validator_spec.loader.exec_module(orion_validator)
orion_validator.validate_wakeword_assets(
    wakeword_path=EXPORT_MODEL_PATH,
    melspectrogram_path=FEATURE_DIR / "melspectrogram.onnx",
    embedding_path=FEATURE_DIR / "embedding_model.onnx",
    manifest_path=MODEL_EXPORT_DIR / "phase13-artifacts.json",
    runtime_version="0.6.0",
)
validation_report["orion_current_validator"] = "PASS"
write_json(WORKSPACE / "eval" / "validation_report.json", validation_report)
print("PASS: uploaded current Orion wakeword_assets validator accepted classifier, feature hashes, manifest, ONNX signatures and CPU smoke runs.")

## 15. Package a Candidate Without Activating It

The bundle is a staging deliverable only. It contains the classifier, Orion manifest, metadata, training configuration, provenance, dataset/license manifests, and evaluation reports. It excludes all training WAVs and downloaded bulk feature arrays. Nothing is copied to the Pi, and no environment/config file is changed.

In [ ]:
if validation_report.get("orion_current_validator") != "PASS":
    raise RuntimeError("Orion validator has not passed; refusing to package a deployable candidate")
EXPORT_DIR = WORKSPACE / "exports" / "hey_orion_v0_6_0"
for directory in (EXPORT_DIR, EXPORT_DIR / "manifests", EXPORT_DIR / "licenses"):
    directory.mkdir(parents=True, exist_ok=True)

for filename in ("hey_orion.onnx", "hey_orion_metadata.json", "phase13-artifacts.json", "training_config.yaml", "provenance.json"):
    source = MODEL_EXPORT_DIR / filename
    if not source.is_file():
        raise FileNotFoundError(f"Required export missing: {source}")
    shutil.copy2(source, EXPORT_DIR / filename)
for source in (WORKSPACE / "eval" / "evaluation.json", WORKSPACE / "eval" / "validation_report.json", WORKSPACE / "provenance" / "requirements_colab.lock", WORKSPACE / "provenance" / "run_log.txt", WORKSPACE / "provenance" / "training_completion.json"):
    shutil.copy2(source, EXPORT_DIR / source.name)
for source in (POSITIVE_MANIFEST, negative_manifest_path, noise_manifest_path, rir_manifest_path, validation_manifest_path):
    shutil.copy2(source, EXPORT_DIR / "manifests" / source.name)
shutil.copy2(WORKSPACE / "licenses" / "license_summary.json", EXPORT_DIR / "licenses" / "license_summary.json")
shutil.copy2(WORKSPACE / "licenses" / "license_summary.md", EXPORT_DIR / "licenses" / "license_summary.md")
(EXPORT_DIR / "licenses" / "license_summary_noise.md").write_text(
    "No third-party raw background/noise recordings were downloaded. Noise was generated by the pinned openWakeWord augmentation code; generic negative features are separately recorded in negative_manifest.csv.\n",
    encoding="utf-8",
)
(EXPORT_DIR / "licenses" / "license_summary_rir.md").write_text(
    "No RIR assets were used. The upstream MIT_environmental_impulse_responses dataset card declares its license unknown, so it was excluded.\n",
    encoding="utf-8",
)

split_report = {
    "positive_train_count": completion["positive_train_count"],
    "positive_validation_count": completion["positive_validation_count"],
    "positive_audio_sha256_intersection": 0,
    "positive_audio_dedup_method": "SHA-256 set intersection of every generated positive train and held-out WAV",
    "generic_negative_train_revision": FEATURE_DATASET_REVISION,
    "false_positive_validation_revision": FEATURE_DATASET_REVISION,
    "generic_train_file_sha256": train_sha,
    "false_positive_validation_file_sha256": validation_sha,
    "generic_train_and_validation_files_differ": train_sha != validation_sha,
    "external_feature_window_dedup": "not independently recomputed; split follows pinned upstream dataset's distinct ACAV100M train and curated false-positive validation sources",
}
write_json(EXPORT_DIR / "manifests" / "split_report.json", split_report)
completion["validation_completed"] = True
completion["orion_validator"] = "PASS"
completion["validation_report_sha256"] = sha256_file(WORKSPACE / "eval" / "validation_report.json")
write_json(EXPORT_DIR / "training_completion.json", completion)

checksum_lines = []
for path in sorted(p for p in EXPORT_DIR.rglob("*") if p.is_file() and p.name != "sha256sums.txt"):
    checksum_lines.append(f"{sha256_file(path)}  {path.relative_to(EXPORT_DIR).as_posix()}")
(EXPORT_DIR / "sha256sums.txt").write_text("\n".join(checksum_lines) + "\n", encoding="utf-8")
BUNDLE_PATH = WORKSPACE / "exports" / "orion_hey_orion_training_bundle.zip"
import zipfile
with zipfile.ZipFile(BUNDLE_PATH, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as archive:
    for path in sorted(p for p in EXPORT_DIR.rglob("*") if p.is_file()):
        archive.write(path, path.relative_to(EXPORT_DIR).as_posix())
print("Candidate bundle ready (not activated):", BUNDLE_PATH)
print("Included files:")
for path in sorted(p for p in EXPORT_DIR.rglob("*") if p.is_file()):
    print(" -", path.relative_to(EXPORT_DIR).as_posix())

In [ ]:
provenance["training_completion"] = completion
provenance["validation_report_sha256"] = sha256_file(EXPORT_DIR / "validation_report.json")
write_json(EXPORT_DIR / "provenance.json", provenance)
checksum_lines = []
for path in sorted(p for p in EXPORT_DIR.rglob("*") if p.is_file() and p.name != "sha256sums.txt"):
    checksum_lines.append(f"{sha256_file(path)}  {path.relative_to(EXPORT_DIR).as_posix()}")
(EXPORT_DIR / "sha256sums.txt").write_text("\n".join(checksum_lines) + "\n", encoding="utf-8")
with zipfile.ZipFile(BUNDLE_PATH, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as archive:
    for path in sorted(p for p in EXPORT_DIR.rglob("*") if p.is_file()):
        archive.write(path, path.relative_to(EXPORT_DIR).as_posix())
assert zipfile.is_zipfile(BUNDLE_PATH)
print("Final provenance and SHA-256 index are inside the bundle.")

## 16. Colab Operator Checklist (A/B/C/D)

1. In Colab choose **Runtime â†’ Change runtime type â†’ T4 GPU** (or another Linux GPU), then set the two license acknowledgements only after reviewing the terms. Upload the two staged feature ONNX files and the two current Orion source files when prompted. Click **Runtime â†’ Run all**. Continue only if the final ORT, feature-chain, and current Orion validator cells print PASS.
2. Download `orion_hey_orion_training_bundle.zip`. It contains `hey_orion.onnx`, `hey_orion_metadata.json`, `phase13-artifacts.json`, `validation_report.json`, `evaluation.json`, `training_config.yaml`, `sha256sums.txt`, `provenance.json`, `training_completion.json`, `requirements_colab.lock`, `manifests/` (also zipped as `manifests.zip`), and `licenses/` (also zipped as `licenses.zip`). It excludes training audio and bulk datasets.
3. Extract the bundle into the Orion repository at `artifacts/wakeword-staging/hey_orion_v0_6_0/`. Keep the already-staged `melspectrogram.onnx` and `embedding_model.onnx` in `artifacts/wakeword-staging/`; do not change systemd or the Pi environment.
4. Install the local optional dependency and run the printed validator command with process-local environment variables only. Remove those variables afterward. Do not enable the profile; VOSK remains active.

In [ ]:
# Add separate manifest/license archives requested for convenient review.
for folder_name in ("manifests", "licenses"):
    archive_path = EXPORT_DIR / f"{folder_name}.zip"
    with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as archive:
        for path in sorted((EXPORT_DIR / folder_name).rglob("*")):
            if path.is_file():
                archive.write(path, path.relative_to(EXPORT_DIR / folder_name).as_posix())

# Copy named deliverables to the Colab Files root as well as keeping the complete bundle.
standalone_names = [
    "hey_orion.onnx", "hey_orion_metadata.json", "validation_report.json",
    "evaluation.json", "training_config.yaml", "sha256sums.txt", "provenance.json",
    "manifests.zip", "licenses.zip",
]
for filename in standalone_names:
    source = EXPORT_DIR / filename if (EXPORT_DIR / filename).is_file() else WORKSPACE / "eval" / filename
    if not source.is_file():
        raise FileNotFoundError(f"Expected downloadable output is missing: {source}")
    shutil.copy2(source, WORKSPACE / "exports" / filename)

# Rebuild checksums and the bundle after adding the standalone archives.
checksum_lines = []
for path in sorted(p for p in EXPORT_DIR.rglob("*") if p.is_file() and p.name != "sha256sums.txt"):
    checksum_lines.append(f"{sha256_file(path)}  {path.relative_to(EXPORT_DIR).as_posix()}")
(EXPORT_DIR / "sha256sums.txt").write_text("\n".join(checksum_lines) + "\n", encoding="utf-8")
shutil.copy2(EXPORT_DIR / "sha256sums.txt", WORKSPACE / "exports" / "sha256sums.txt")
with zipfile.ZipFile(BUNDLE_PATH, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as archive:
    for path in sorted(p for p in EXPORT_DIR.rglob("*") if p.is_file()):
        archive.write(path, path.relative_to(EXPORT_DIR).as_posix())
assert zipfile.is_zipfile(BUNDLE_PATH)

print("A. Runtime â†’ Change runtime type â†’ choose Linux GPU; review both license gates; upload the two staged ONNX files plus local wakeword_assets.py and config.py; Runtime â†’ Run all.")
print("B. Download one archive:", BUNDLE_PATH.name, "(contains the ONNX, metadata, manifest, reports, config, checksums, provenance, manifests.zip, licenses.zip).")
print("C. Extract to: artifacts/wakeword-staging/hey_orion_v0_6_0/")
print("   Keep shared feature graphs at artifacts/wakeword-staging/melspectrogram.onnx and embedding_model.onnx")
print("D. In Orion repository PowerShell, run this process-local validation sequence; it does not persist config or touch the Pi:")
print(r'''Set-Location <ORION-repository>
.\.venv-local\Scripts\python.exe -m pip install -e ".[wakeword]"
$env:ORION_VOICE_PROFILE = "wakeword"
$env:ORION_STT_BACKEND = "whisper.cpp"
$env:ORION_WAKE_WORD_MODEL_PATH = (Resolve-Path ".\artifacts\wakeword-staging\hey_orion_v0_6_0\hey_orion.onnx").Path
$env:ORION_WAKE_WORD_MELSPECTROGRAM_PATH = (Resolve-Path ".\artifacts\wakeword-staging\melspectrogram.onnx").Path
$env:ORION_WAKE_WORD_EMBEDDING_PATH = (Resolve-Path ".\artifacts\wakeword-staging\embedding_model.onnx").Path
$env:ORION_WAKE_WORD_MANIFEST_PATH = (Resolve-Path ".\artifacts\wakeword-staging\hey_orion_v0_6_0\phase13-artifacts.json").Path
.\.venv-local\Scripts\python.exe -m orion --validate-wakeword-assets
'ORION_VOICE_PROFILE','ORION_STT_BACKEND','ORION_WAKE_WORD_MODEL_PATH','ORION_WAKE_WORD_MELSPECTROGRAM_PATH','ORION_WAKE_WORD_EMBEDDING_PATH','ORION_WAKE_WORD_MANIFEST_PATH' | ForEach-Object { Remove-Item "Env:$_" -ErrorAction SilentlyContinue }''')
from google.colab import files
files.download(str(BUNDLE_PATH))